In [1]:
import sys
sys.path.insert(0, '..')

## 1. Setup

In [2]:
from datetime import date, timedelta
from app.config import get_settings
from clients.news_api import FinnhubNewsClient, MarketAuxNewsClient, get_news_client

settings = get_settings()
print(f"Finnhub key set  : {bool(settings.finnhub_api_key)}")
print(f"MarketAux key set: {bool(settings.marketaux_api_key)}")
print(f"Configured sources: {settings.news_sources}")

TICKER    = "NVDA"
to_date   = date.today()
from_date = to_date - timedelta(days=7)
print(f"\nDate range: {from_date} → {to_date}")

Finnhub key set  : True
MarketAux key set: True
Configured sources: ['finnhub']

Date range: 2026-04-17 → 2026-04-24


## 2. Finnhub

In [9]:
async with FinnhubNewsClient(api_key=settings.finnhub_api_key) as client:
    finnhub_articles = await client.get_news(TICKER, to_date, to_date)

print(f"Finnhub: {len(finnhub_articles)} articles")
for a in finnhub_articles[:3]:
    print(f"  [{a.published_at.strftime('%Y-%m-%d %H:%M')}] {a.source}")
    print(f"  {a.headline}")

Finnhub: 139 articles
  [2026-04-24 19:53] Yahoo
  Here's What to Do if the Market Crashes Right After You Retire
  [2026-04-24 19:31] Yahoo
  Kalshi Taps Pyth Network To Settle Bets On Gold, Oil, Wheat And More
  [2026-04-24 19:26] Yahoo
  The U.S. is up $26 billion on its Intel trade


In [11]:
from collections import Counter

source_counts = Counter(a.source for a in finnhub_articles)
print(f"{len(source_counts)} unique sources from Finnhub:\n")
for source, count in source_counts.most_common():
    print(f"  {count:3}  {source}")

6 unique sources from Finnhub:

  116  Yahoo
    9  Benzinga
    6  SeekingAlpha
    4  CNBC
    2  ChartMill
    2  Finnhub


## 3. MarketAux

In [10]:
async with MarketAuxNewsClient(api_key=settings.marketaux_api_key) as client:
    marketaux_articles = await client.get_news(TICKER, to_date, to_date)

print(f"MarketAux: {len(marketaux_articles)} articles")
for a in marketaux_articles[:3]:
    print(f"  [{a.published_at.strftime('%Y-%m-%d %H:%M')}] {a.source}")
    print(f"  {a.headline}")

MarketAux: 3 articles
  [2026-04-24 19:15] insidermonkey.com
  Is Vicor (VICR) One of the Top NASDAQ Growth Stocks to Buy and Hold Forever?
  [2026-04-24 19:11] etftrends.com
  Why Free Cash Flow Should Be Front of Mind in Today’s Market
  [2026-04-24 18:07] thestockmarketwatch.com
  Tech and Semiconductors Surge as Nasdaq Gains Over 1.4%; Energy Lags Amid Oil Price Slump


In [4]:
marketaux_articles

[NewsArticle(ticker='NVDA', headline='Is Vicor (VICR) One of the Top NASDAQ Growth Stocks to Buy and Hold Forever?', summary='Vicor Corporation (NASDAQ:VICR) is one of the best NASDAQ growth stocks to buy and hold forever.', source='insidermonkey.com', url='https://www.insidermonkey.com/blog/is-vicor-vicr-one-of-the-top-nasdaq-growth-stocks-to-buy-and-hold-forever-1746148/', published_at=datetime.datetime(2026, 4, 24, 19, 15, 45, tzinfo=datetime.timezone.utc)),
 NewsArticle(ticker='NVDA', headline='Why Free Cash Flow Should Be Front of Mind in Today’s Market', summary='Exchange 2026: Michael Mack reveals why free cash flow is the "weighing machine" for 2026. VFLO and GFLW avoid concentration risks.', source='etftrends.com', url='https://www.etftrends.com/free-cash-flow-content-hub/why-free-cash-flow-ideal-play-todays-market/', published_at=datetime.datetime(2026, 4, 24, 19, 11, 34, tzinfo=datetime.timezone.utc)),
 NewsArticle(ticker='NVDA', headline='Tech and Semiconductors Surge as Na

## 4. Aggregated (both sources, deduplicated)

In [12]:
api_keys = settings.news_api_keys()
async with get_news_client(["finnhub", "marketaux"], api_keys) as client:
    all_articles = await client.get_news(TICKER, from_date, to_date)

print(f"Aggregated: {len(all_articles)} articles (Finnhub: {len(finnhub_articles)}, MarketAux: {len(marketaux_articles)})")
for a in all_articles[:5]:
    print(f"  [{a.published_at.strftime('%Y-%m-%d %H:%M')}] [{a.source}] {a.headline}")

Aggregated: 251 articles (Finnhub: 139, MarketAux: 3)
  [2026-04-24 19:53] [Yahoo] Here's What to Do if the Market Crashes Right After You Retire
  [2026-04-24 19:31] [Yahoo] Kalshi Taps Pyth Network To Settle Bets On Gold, Oil, Wheat And More
  [2026-04-24 19:26] [Yahoo] The U.S. is up $26 billion on its Intel trade
  [2026-04-24 19:25] [Yahoo] Uber and Joby Aviation Confirm Air Taxis Launch in Dubai by 2026 -- Time to Buy?
  [2026-04-24 19:21] [Yahoo] Meta Layoffs: Is the Facebook Parent Getting Ready for Another "Year of Efficiency"?


## 5. Headlines for NewsAgent

In [13]:
headlines = [a.headline for a in all_articles]
print(f"{len(headlines)} headlines ready for NewsAgent:")
for h in headlines[:10]:
    print(f"  - {h}")

251 headlines ready for NewsAgent:
  - Here's What to Do if the Market Crashes Right After You Retire
  - Kalshi Taps Pyth Network To Settle Bets On Gold, Oil, Wheat And More
  - The U.S. is up $26 billion on its Intel trade
  - Uber and Joby Aviation Confirm Air Taxis Launch in Dubai by 2026 -- Time to Buy?
  - Meta Layoffs: Is the Facebook Parent Getting Ready for Another "Year of Efficiency"?
  - NVIDIA (NVDA) Partners With Oklo, LANL to Develop Nuclear-Powered AI Factories
  - Is Vicor (VICR) One of the Top NASDAQ Growth Stocks to Buy and Hold Forever?
  - Why Northrup Grumman Stock Sank This Week
  - Why Free Cash Flow Should Be Front of Mind in Today’s Market
  - Super Micro Computer Stock Jumps 9% Despite Oracle Canceling $1.4B Contract: What’s Really Driving the Move?


In [18]:
keywords = ['NVDA', 'NVIDIA']
relevant = [hl for hl in headlines if any(kw.lower() in hl.lower() for kw in keywords)]

In [19]:
relevant

['NVIDIA (NVDA) Partners With Oklo, LANL to Develop Nuclear-Powered AI Factories',
 'Nvidia market cap back above $5T, Alphabet to invest $40B in Anthropic',
 'Nvidia, Intel and AMD Lead Chip Stocks as AI Wave Lifts Sector',
 'Cadence’s Expanded AI Alliances With TSMC, NVIDIA and Google Might Change The Case For Investing In Cadence Design Systems (CDNS)',
 'Nvidia stock on track for record close, pushing market cap over $5 trillion once again',
 'Stock Market Today: Nasdaq, S&P 500 Rise As Intel Surges; Nvidia Soars (Live Coverage)',
 'Why Nvidia Stock Popped Today',
 'Nvidia Expands AI Use with Early GPT-5.5 Access',
 "Adobe Unleashed: Why Nvidia's CEO Says This AI Giant Is the Future of Marketing",
 'Nvidia Stock Races Toward Record Closing High',
 'Nvidia Is Rallying. Why Intel, Hyperscalers May Be The Reason.',
 'NIO develops in-house AI chips to cut reliance on Nvidia',
 'Chinese EV maker NIO bets on in-house chips to cut reliance on Nvidia',
 'Why Isn’t NVIDIA Stock at $300 Whil

In [20]:
len(relevant)

39